# AdventureWorks on Lakebase

Predict over the AdventureWorks sample in a Lakebase database, where its tables already sit.
Lakebase is PostgreSQL, so the client samples the tables with SQL in place: describing the schema reads a preview of each, and after that only the rows a prediction needs leave the database.

You need the four AdventureWorks tables (`adventureworks_customers`, `adventureworks_products`, `adventureworks_sales_order_headers`, `adventureworks_sales_order_details`) in one schema of a Lakebase database, a Lakebase role for the notebook's user, and a Kumo Relational model on a Model Serving endpoint.

Run it on serverless compute, which reaches Lakebase privately.
A classic cluster reaches it over the public internet, which works only if the workspace lets that cluster's egress address in; a workspace with IP access lists may not.


In [ ]:
dbutils.widgets.text(
    'install',
    # The `lakebase` extra, not `postgres`: it brings plain psycopg, which uses
    # the runtime's own libpq, where the `postgres` extra's psycopg[binary]
    # bundles an OpenSSL that fails its FIPS self-test on Databricks' aarch64
    # machines and aborts the process. Bound a version with `~=`, not `>=`: a
    # classic cluster's %pip reads this line through a shell, which takes `>`
    # as a redirection.
    'structured-data-models-client[relational,lakebase,databricks-serving]',
    'pip install',
)
dbutils.widgets.text('lakebase_endpoint', '', 'Lakebase endpoint resource')
dbutils.widgets.text('database', 'databricks_postgres', 'Database')
dbutils.widgets.text('schema', 'adventureworks', 'Schema')
dbutils.widgets.text('endpoint', '', 'Model Serving endpoint')

install = dbutils.widgets.get('install')

# The runtime's packages are built against the NumPy it ships; pinning it keeps
# the install from upgrading NumPy beneath them, which stops Python starting.
import numpy

keep_numpy = f'numpy=={numpy.__version__}'

In [ ]:
%pip install --only-binary=:all: $install $keep_numpy

In [ ]:
dbutils.library.restartPython()

In [ ]:
import json
import os
import platform
import sys
from importlib.metadata import PackageNotFoundError, version


def installed(package):
    try:
        return version(package)
    except PackageNotFoundError:
        return None


ENVIRONMENT = {
    'runtime': os.environ.get('DATABRICKS_RUNTIME_VERSION'),
    'python': sys.version.split()[0],
    'machine': platform.machine(),
    **{
        package: installed(package)
        for package in (
            'structured-data-models-client',
            'structured-data-models-relational-engine',
            'numpy',
            'pandas',
            'pyarrow',
            'pydantic',
            'databricks-sdk',
            'databricks-sql-connector',
            'psycopg',
        )
    },
}
print(json.dumps(ENVIRONMENT, indent=1))

## 1. Say where the tables are

Lakebase signs in with a short-lived OAuth token as the password, minted for the notebook's user.
It is checked when the connection opens, so an open connection outlives it.
`lakebase()` mints it with the notebook's own `WorkspaceClient`, and looks up the endpoint's host.

In [ ]:
from databricks.sdk import WorkspaceClient
from sdm_connectors import PostgresSource
from sdm_connectors.databricks import lakebase

lakebase_endpoint = dbutils.widgets.get('lakebase_endpoint')
database = dbutils.widgets.get('database')
schema_name = dbutils.widgets.get('schema')
endpoint = dbutils.widgets.get('endpoint')
for value, name in (
    (lakebase_endpoint, 'lakebase_endpoint'),
    (endpoint, 'endpoint'),
):
    if not value:
        raise ValueError(f'Set the {name} widget.')

workspace = WorkspaceClient()
connection = lakebase(lakebase_endpoint, database=database, workspace=workspace)
source = PostgresSource(connection, schema=schema_name)
source

## 2. Describe the tables

This database declares its primary and foreign keys, so `Schema.from_source` reads them from the catalog: nothing about keys or links is written here.

In [ ]:
from sdm_client import Schema

schema = Schema.from_source(
    source,
    {
        'customers': 'adventureworks_customers',
        'products': 'adventureworks_products',
        'sales_order_headers': 'adventureworks_sales_order_headers',
        'sales_order_details': 'adventureworks_sales_order_details',
    },
    time_columns={
        'sales_order_headers': 'OrderDate',
        'sales_order_details': 'OrderDate',
    },
)

Inference leaves customers without a time column, because its only date records when a row was last changed.
Inference takes `SellStartDate` for products, but a product's sales dates say when it was on sale rather than when anything happened to it, so the next cell clears it.

`rowguid` is a per-row UUID AdventureWorks carries for replication: it identifies a row and says nothing about it, so it is left out.

In [ ]:
schema['products'].time_column = None
for table in schema.tables.values():
    if 'rowguid' in table:
        del table['rowguid']

for table in schema.tables.values():
    print(table)
print(schema.relationships)

## 3. Ask

The same questions as the Unity Catalog notebook: demand now, as of six months earlier, and three windows out; which frequent customers go quiet; what one customer buys next; and a product's colour.

In [ ]:
import pandas as pd


def rows(query):
    with connection.cursor() as cursor:
        cursor.execute(query)
        return cursor.fetchall()


def table(name):
    return f'"{schema_name}"."adventureworks_{name}"'


top_product = rows(
    f'SELECT "ProductID" FROM {table("sales_order_details")} '
    'GROUP BY "ProductID" ORDER BY sum("OrderQty") DESC, "ProductID" LIMIT 1'
)[0][0]
# Bought by the most customers, so each has buyers in the context.
popular_products = [
    row[0]
    for row in rows(
        f'SELECT "ProductID" FROM {table("sales_order_details")} '
        'GROUP BY "ProductID" ORDER BY count(DISTINCT "CustomerID") DESC, "ProductID" LIMIT 10'
    )
]
frequent_customers = [
    row[0]
    for row in rows(
        f'SELECT "CustomerID" FROM {table("sales_order_headers")} '
        'GROUP BY "CustomerID" ORDER BY count(*) DESC, "CustomerID" LIMIT 25'
    )
]
latest = rows(f'SELECT max("OrderDate") FROM {table("sales_order_details")}')[
    0
][0]
anchor = pd.Timestamp(latest) - pd.Timedelta(days=180)
print(
    f'top product {top_product}; latest order {latest}; anchor {anchor.date()}'
)

In [ ]:
# Four estimators over shuffled columns: one estimator's answer moves with
# column order, and averaging over four permutations settles it.
ENSEMBLE = {'num_estimators': 4, 'column_shuffle': True}
DEMAND = (
    'PREDICT SUM(sales_order_details.OrderQty, 0, 30, days) '
    'FOR EACH products.ProductID'
)

QUESTIONS = {
    'demand_30d': (
        DEMAND,
        {'indices': [top_product], 'inference_config': ENSEMBLE},
    ),
    'demand_30d_historical': (
        DEMAND,
        {
            'indices': [top_product],
            'anchor_time': anchor,
            'inference_config': ENSEMBLE,
        },
    ),
    'demand_3_timeframes': (
        'PREDICT SUM(sales_order_details.OrderQty, 0, 30, days) '
        'FORECAST 3 TIMEFRAMES FOR EACH products.ProductID',
        {'indices': [top_product], 'anchor_time': anchor},
    ),
    'inactivity_90d': (
        'PREDICT COUNT(sales_order_headers.*, 0, 90, days)=0 '
        'FOR EACH customers.CustomerID',
        {'indices': frequent_customers},
    ),
    'recommend_top5': (
        'PREDICT LIST_DISTINCT(sales_order_details.ProductID, 0, 30, days) '
        'RANK TOP 5 FOR EACH customers.CustomerID',
        {
            'indices': frequent_customers[:1],
            'candidates': popular_products,
            'anchor_time': anchor,
            'num_context_samples': 1000,
        },
    ),
    'product_color': (
        'PREDICT products.Color FOR EACH products.ProductID',
        {'indices': [top_product]},
    ),
}

In [ ]:
from sdm_client import (
    KumoRelational,
    PredictiveQuery,
    SDMClient,
    StructuredDataError,
)

# A question the data cannot answer is refused with the reason, and the rest
# are still asked.
results, refused = {}, {}
with SDMClient.for_databricks_serving(endpoint) as client:
    for name, (text, options) in QUESTIONS.items():
        try:
            request = PredictiveQuery(text, schema).materialize(**options)
            answer = client.predict(request, model=KumoRelational())
        except StructuredDataError as error:
            refused[name] = str(error)
            print(f'{name}: refused – {error}')
            continue
        results[name] = answer.to_pandas()
        print(f'{name}: {len(results[name])} rows')
        display(results[name])

In [ ]:
dbutils.notebook.exit(
    json.dumps(
        {
            'environment': ENVIRONMENT,
            'results': {
                name: frame.to_dict('records')
                for name, frame in results.items()
            },
            'refused': refused,
        },
        default=str,
    )
)